In [2]:
import os
import duckdb
import pandas as pd

# Redireciona o diretório de execução para a raiz do projeto
if os.getcwd().endswith('notebooks'):
  os.chdir('..')

# Conexão OLAP em memória
con = duckdb.connect()

# 1. Visão geral do volume das tabelas
df_volumes = con.execute("""
    SELECT 'application_train' AS tabela, COUNT(*) AS total_registros FROM 'data/application_train.parquet'
    UNION ALL
    SELECT 'bureau', COUNT(*) FROM 'data/bureau.parquet'
    UNION ALL
    SELECT 'previous_application', COUNT(*) FROM 'data/previous_application.parquet'
    UNION ALL
    SELECT 'pos_cash_balance', COUNT(*) FROM 'data/pos_cash_balance.parquet'
""").df()

print("--- Volume de Dados ---")
display(df_volumes)

# 2. Taxa de inadimplência (TARGET) e métricas financeiras básicas
df_target = con.execute("""
    SELECT 
        NAME_CONTRACT_TYPE AS tipo_contrato,
        COUNT(*) AS total_clientes,
        SUM(TARGET) AS inadimplentes,
        ROUND(AVG(TARGET) * 100, 2) AS taxa_inadimplencia_pct,
        ROUND(AVG(AMT_INCOME_TOTAL), 2) AS renda_media,
        ROUND(AVG(AMT_CREDIT), 2) AS credito_medio,
        ROUND(AVG(AMT_CREDIT / AMT_INCOME_TOTAL), 2) AS raz_credito_renda
    FROM 'data/application_train.parquet'
    GROUP BY NAME_CONTRACT_TYPE
""").df()

print("\n--- Perfil do Contrato vs Inadimplência ---")
display(df_target)

--- Volume de Dados ---


,tabela,total_registros
0,application_train,307511
1,bureau,1716428
2,previous_application,1670214
3,pos_cash_balance,10001358



--- Perfil do Contrato vs Inadimplência ---


,tipo_contrato,total_clientes,inadimplentes,taxa_inadimplencia_pct,renda_media,credito_medio,raz_credito_renda
0,Revolving loans,29279,1604.0,5.48,166217.02,324017.98,2.15
1,Cash loans,278232,23221.0,8.35,169069.51,627965.73,4.15


### Passo 1: Consolidação no DuckDB (Visão Contrato e Visão Cliente)

In [3]:
import duckdb
import numpy as np
import pandas as pd

con = duckdb.connect()

# 1. VISÃO CONTRATO: Agregações relacionais do Bureau e Apólices Anteriores
con.execute("""
    CREATE OR REPLACE VIEW vw_bureau_contrato AS
    SELECT 
        SK_ID_CURR,
        COUNT(SK_ID_BUREAU) AS bureau_qtd_creditos,
        COUNT(CASE WHEN CREDIT_ACTIVE = 'Active' THEN 1 END) AS bureau_qtd_ativos,
        ROUND(SUM(COALESCE(AMT_CREDIT_SUM, 0)), 2) AS bureau_total_credito,
        ROUND(SUM(COALESCE(AMT_CREDIT_SUM_DEBT, 0)), 2) AS bureau_total_divida
    FROM 'data/bureau.parquet'
    GROUP BY SK_ID_CURR;
""")

con.execute("""
    CREATE OR REPLACE VIEW vw_prev_app_contrato AS
    SELECT 
        SK_ID_CURR,
        COUNT(SK_ID_PREV) AS prev_qtd_pedidos,
        COUNT(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1 END) AS prev_qtd_aprovados,
        COUNT(CASE WHEN NAME_CONTRACT_STATUS = 'Refused' THEN 1 END) AS prev_qtd_recusados
    FROM 'data/previous_application.parquet'
    GROUP BY SK_ID_CURR;
""")

# 2. VISÃO CLIENTE + PASSO 1: Consolidação e Limpeza de Anomalias
df_visao_cliente = con.execute("""
    SELECT 
        app.SK_ID_CURR,
        app.TARGET,
        app.NAME_CONTRACT_TYPE,
        app.CODE_GENDER,
        app.FLAG_OWN_CAR,
        app.FLAG_OWN_REALTY,
        app.AMT_INCOME_TOTAL,
        app.AMT_CREDIT,
        app.AMT_ANNUITY,
        
        -- Tratamento de Anomalia do Kaggle (365243 dias = ~1000 anos -> Nulo)
        CASE WHEN app.DAYS_EMPLOYED = 365243 THEN NULL ELSE app.DAYS_EMPLOYED END AS DAYS_EMPLOYED,
        app.DAYS_BIRTH,
        
        -- Joins das Features Visão Contrato
        COALESCE(b.bureau_qtd_creditos, 0) AS bureau_qtd_creditos,
        COALESCE(b.bureau_qtd_ativos, 0) AS bureau_qtd_ativos,
        COALESCE(b.bureau_total_credito, 0) AS bureau_total_credito,
        COALESCE(b.bureau_total_divida, 0) AS bureau_total_divida,
        COALESCE(p.prev_qtd_pedidos, 0) AS prev_qtd_pedidos,
        COALESCE(p.prev_qtd_aprovados, 0) AS prev_qtd_aprovados,
        COALESCE(p.prev_qtd_recusados, 0) AS prev_qtd_recusados
        
    FROM 'data/application_train.parquet' app
    LEFT JOIN vw_bureau_contrato b ON app.SK_ID_CURR = b.SK_ID_CURR
    LEFT JOIN vw_prev_app_contrato p ON app.SK_ID_CURR = p.SK_ID_CURR
""").df()

print(f"Base Visão Cliente gerada com sucesso! Formato: {df_visao_cliente.shape}")

Base Visão Cliente gerada com sucesso! Formato: (307511, 18)


### Passo 2: Pré-processamento e Seleção de Features via OptBinning (WoE / IV)

In [5]:
from optbinning import BinningProcess

(CVXPY) Sep 14 09:10:26 PM: Encountered unexpected exception importing solver HIGHS:
ImportError('/usr/local/python/3.14.2/lib/python3.14/site-packages/highspy/_core.cpython-314-x86_64-linux-gnu.so: undefined symbol: _ZN5Highs13releaseMemoryEv')


In [6]:

# Definição das variáveis explicativas e target
X = df_visao_cliente.drop(columns=["SK_ID_CURR", "TARGET"])
y = df_visao_cliente["TARGET"]

categorical_features = list(X.select_dtypes(include=["object"]).columns)
variable_names = list(X.columns)

# Inicializa o BinningProcess do OptBinning
binning_process = BinningProcess(
    variable_names=variable_names,
    categorical_variables=categorical_features,
)

# Ajusta WoE / IV
binning_process.fit(X, y)

# Tabela resumo de Information Value (IV)
df_iv = binning_process.summary()
display(df_iv.sort_values(by="iv", ascending=False).head(10))

/tmp/ipykernel_25502/206079627.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = list(X.select_dtypes(include=["object"]).columns)


,name,dtype,status,selected,n_bins,iv,js,gini,quality_score
7,DAYS_EMPLOYED,numerical,OPTIMAL,True,9,0.114054,0.014145,0.187445,0.352536
8,DAYS_BIRTH,numerical,OPTIMAL,True,14,0.087245,0.010818,0.166474,0.083814
5,AMT_CREDIT,numerical,OPTIMAL,True,9,0.059368,0.007361,0.131391,0.180589
15,prev_qtd_recusados,numerical,OPTIMAL,True,4,0.053781,0.006675,0.111268,0.149919
1,CODE_GENDER,categorical,OPTIMAL,True,2,0.038601,0.004816,0.095251,0.144002
6,AMT_ANNUITY,numerical,OPTIMAL,True,9,0.031179,0.003883,0.09482,0.121474
10,bureau_qtd_ativos,numerical,OPTIMAL,True,4,0.02549,0.003166,0.065628,0.063302
11,bureau_total_credito,numerical,OPTIMAL,True,7,0.020528,0.002559,0.070208,0.005989
9,bureau_qtd_creditos,numerical,OPTIMAL,True,8,0.015569,0.001943,0.065525,0.03218
0,NAME_CONTRACT_TYPE,categorical,OPTIMAL,True,2,0.015039,0.001868,0.033288,0.021574
